In [1]:
import os
import json
import pickle
import pandas as pd
import random
import numpy as np

# Load mapping

In [2]:
mapping = json.load(open("kgindex.json"))
entities = dict([(value, key) for key, value in mapping["e"].items()])
relations = \
    {0: 'is related to',
     1: 'has context of',
     2: 'is a',
     3: 'is synonym of',
     4: 'at location',
     5: 'is etymologically related to',
     6: 'is distinct from',
     7: 'has the last subevent',
     8: 'is used for',
     9: 'is similar to',
     10: 'desires',
     11: 'is antonym of',
     12: 'dbpedia',
     13: 'is a part of',
     14: 'is a form of',
     15: 'has a',
     16: 'is capable of',
     17: 'is an instance of',
     18: 'has a prerequisite of',
     19: 'is motivated by the goal of',
     20: 'is derived from',
     21: 'has subevent',
     22: 'causes',
     23: 'receives sanction by',
     24: 'has the property of',
     25: 'entails',
     26: 'has the first subevent',
     27: 'does not desire',
     28: 'causes desire',
     29: 'is made of',
     30: 'does not have the property of',
     31: 'is created by',
     32: 'is located near',
     33: 'is not capable of',
     34: 'is defined as',
     35: 'is a manner of'}

# Load Pickle files

In [3]:
path_profix = "llm"

pickle_files = {}
for t in ["type0000",
          "type0001",
          "type0002",
          "type0003",
          "type0004",
          "type0005",
          "type0006",
          "type0007",
          "type0008",
          "type0009",
          "type0010",
          "type0011"]:
    path_to_pickle = path_profix + "/" + t + ".pickle"
    if os.path.isfile(path_to_pickle):
        if t not in pickle_files:
            pickle_files[t] = [path_to_pickle]
        else:
            pickle_files[t].append(path_to_pickle)
pickle_files

{'type0000': ['llm/type0000.pickle'],
 'type0001': ['llm/type0001.pickle'],
 'type0002': ['llm/type0002.pickle'],
 'type0003': ['llm/type0003.pickle'],
 'type0004': ['llm/type0004.pickle'],
 'type0005': ['llm/type0005.pickle'],
 'type0006': ['llm/type0006.pickle'],
 'type0007': ['llm/type0007.pickle'],
 'type0008': ['llm/type0008.pickle'],
 'type0009': ['llm/type0009.pickle'],
 'type0010': ['llm/type0010.pickle'],
 'type0011': ['llm/type0011.pickle']}

In [4]:
for key, value in pickle_files.items():
    print(value)
    test = open(value[0],"rb")
    # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
    print(pickle.load(test)[:5][0][1])

['llm/type0000.pickle']
r1(s1,f1)
['llm/type0001.pickle']
(r1(s1,e1))&(r2(e1,f1))
['llm/type0002.pickle']
(r1(s1,f1))&(r2(s2,f1))
['llm/type0003.pickle']
(!(r1(s1,f1)))&(r2(s2,f1))
['llm/type0004.pickle']
(r1(s1,f1))&(r2(e1,f1))
['llm/type0005.pickle']
(r1(s1,e1))&((r2(e1,f1))&(r3(e1,f1)))
['llm/type0006.pickle']
(r1(s1,f1))|(r2(s2,f1))
['llm/type0007.pickle']
(!(r1(s1,f1)))&((r2(s2,f1))&(r3(s3,f1)))
['llm/type0008.pickle']
(r1(s1,e1))&((r2(s2,e1))&(r3(e1,f1)))
['llm/type0009.pickle']
(r1(s1,e1))&((r2(s2,e1))&((r3(e1,f1))&(r4(e1,f1))))
['llm/type0010.pickle']
(!(r1(s1,e1)))&((r2(s2,e1))&(r3(e1,f1)))
['llm/type0011.pickle']
(r1(s1,f1))|((r2(s2,e1))&(r3(e1,f1)))


In [5]:
candidate_str = """Four candidate entities: {choice1}, {choice2}, {choice3}, {choice4}\n"""
type0000_str = """Soft query: ({s1}, {r1}, f1, {a1}, {b1})\n"""
type0001_str = """Soft query: ({s1}, {r1}, e1, {a1}, {b1}) \land (e1, {r2}, f1, {a2}, {b2})\n"""
type0002_str = """Soft query: ({s1}, {r1}, f1, {a1}, {b1}) \land ({s2}, {r2}, f1, {a2}, {b2})\n"""

type0003_str = """Soft query: (\neg ({s1}, {r1}, f1, {a1}, {b1})) \land ({s2}, {r2}, f1, {a2}, {b2})\n"""

type0004_str = """Soft query: ({s1}, {r1}, f1, {a1}, {b1}) \land (e1, {r2}, f1, {a2}, {b2})\n"""
type0005_str = """Soft query: ({s1}, {r1}, e1, {a1}, {b1}) \land (e1, {r2}, f1, {a2}, {b2}) \land (e1, {r3}, f1, {a3}, {b3})\n"""

type0006_str = """Soft query: ({s1}, {r1}, f1, {a1}, {b1}) \lor ({s2}, {r2}, f1, {a2}, {b2})\n"""
type0007_str = """Soft query: (\neg ({s1}, {r1}, f1, {a1}, {b1})) \land ({s2}, {r2}, f1, {a2}, {b2}) \land ({s3}, {r3}, f1, {a3}, {b3})\n"""

type0008_str = """Soft query: ({s1}, {r1}, e1, {a1}, {b1}) \land ({s2}, {r2}, e1, {a2}, {b2}) \land (e1, {r3}, f1, {a3}, {b3})\n"""
type0009_str = """Soft query: ({s1}, {r1}, e1, {a1}, {b1}) \land ({s2}, {r2}, e1, {a2}, {b2}) \land (e1, {r3}, f1, {a3}, {b3}) \land (e1, {r4}, f1, {a4}, {b4})\n"""

type0010_str = """Soft query: (\neg ({s1}, {r1}, e1, {a1}, {b1})) \land ({s2}, {r2}, e1, {a2}, {b2}) \land (e1, {r3}, f1, {a3}, {b3})\n"""
type0011_str = """Soft query: ({s1}, {r1}, f1, {a1}, {b1}) \lor (({s2}, {r2}, e1, {a2}, {b2}) \land (e1, {r3}, f1, {a3}, {b3}))\n"""


In [6]:
type0000_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0001_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0002_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0003_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0004_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0005_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0006_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0007_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0008_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0009_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0010_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])
type0011_dataset = pd.DataFrame(columns=['raw', "solution", 'query'])

In [7]:
for t in ["type0000",
          "type0001",
          "type0002",
          "type0003",
          "type0004",
          "type0005",
          "type0006",
          "type0007",
          "type0008",
          "type0009",
          "type0010",
          "type0011"]:
    curr_dataset = eval(t + "_dataset")
    curr_dataset.loc[len(curr_dataset)] = {'raw': "(id, origin_string_query_type, dictionary, [4 candidates], [4 f1_values], correct_anwerer(max_candi))", 'solution': "the expected answer", 'query': "the generated question for human experts"} 


In [8]:
pickle_files

{'type0000': ['llm/type0000.pickle'],
 'type0001': ['llm/type0001.pickle'],
 'type0002': ['llm/type0002.pickle'],
 'type0003': ['llm/type0003.pickle'],
 'type0004': ['llm/type0004.pickle'],
 'type0005': ['llm/type0005.pickle'],
 'type0006': ['llm/type0006.pickle'],
 'type0007': ['llm/type0007.pickle'],
 'type0008': ['llm/type0008.pickle'],
 'type0009': ['llm/type0009.pickle'],
 'type0010': ['llm/type0010.pickle'],
 'type0011': ['llm/type0011.pickle']}

In [9]:
for key, value in pickle_files.items():
    print(value)
    raw_dataset = pickle.load(open(value[0],"rb"))
    curr_type = value[0][4:12]

    for row in raw_dataset:
        # format: (id, origin_string_query_type, dictionary, [4 candidates], [4 f1_values], correct_anwerer(max_candi))
        idx = row[0]

        mapped_values = {}
        mapped_candidates = {}
        
        for p, v in row[2].items():
            if p[0] in ["a", "b"]:
                mapped_values[p] = v
            elif p[0] == "r":
                mapped_values[p] = relations[v]
            else:
                mapped_values[p] = entities[v]

        for c in range(len(row[3])):
            mapped_candidates["choice" + str(c+1)] =  entities[row[3][c]]
            
        input_prompt = eval(curr_type + "_str").format(**mapped_values) + candidate_str.format(**mapped_candidates)
        expected_answer = entities[row[5]]
        
        curr_dataset = eval(curr_type + "_dataset")
        curr_dataset.loc[len(curr_dataset)] = {'raw': row, 'solution': expected_answer, 'query': input_prompt} 


['llm/type0000.pickle']
['llm/type0001.pickle']
['llm/type0002.pickle']
['llm/type0003.pickle']
['llm/type0004.pickle']
['llm/type0005.pickle']
['llm/type0006.pickle']
['llm/type0007.pickle']
['llm/type0008.pickle']
['llm/type0009.pickle']
['llm/type0010.pickle']
['llm/type0011.pickle']


In [10]:
for key, value in pickle_files.items():
    raw_dataset = pickle.load(open(value[0],"rb"))
    curr_type = value[0][4:12]
    print(curr_type + "_dataset" + f".to_excel(writer, sheet_name='{curr_type}')")

type0000_dataset.to_excel(writer, sheet_name='type0000')
type0001_dataset.to_excel(writer, sheet_name='type0001')
type0002_dataset.to_excel(writer, sheet_name='type0002')
type0003_dataset.to_excel(writer, sheet_name='type0003')
type0004_dataset.to_excel(writer, sheet_name='type0004')
type0005_dataset.to_excel(writer, sheet_name='type0005')
type0006_dataset.to_excel(writer, sheet_name='type0006')
type0007_dataset.to_excel(writer, sheet_name='type0007')
type0008_dataset.to_excel(writer, sheet_name='type0008')
type0009_dataset.to_excel(writer, sheet_name='type0009')
type0010_dataset.to_excel(writer, sheet_name='type0010')
type0011_dataset.to_excel(writer, sheet_name='type0011')


# Merge existing data

In [11]:
# def replace_begining(query):
#     return query.replace("Q", "q", 1)

# exist_labeled_data = pd.read_excel(open('Data Annotation v2.xlsx', 'rb'), sheet_name="type0000")  
# exist_labeled_data["query"] = exist_labeled_data["query"].map(replace_begining)
# type0000_dataset = pd.concat([exist_labeled_data, type0000_dataset], ignore_index=True, sort=False).drop_duplicates(subset=['query'])


# exist_labeled_data = pd.read_excel(open('Data Annotation v2.xlsx', 'rb'), sheet_name="type0002")  
# exist_labeled_data["query"] = exist_labeled_data["query"].map(replace_begining)
# type0002_dataset = pd.concat([exist_labeled_data, type0002_dataset], ignore_index=True, sort=False).drop_duplicates(subset=['query'])


# Write to an Excel file

In [12]:
with pd.ExcelWriter('manual.xlsx', engine='xlsxwriter') as writer:
    type0000_dataset.to_excel(writer, sheet_name='type0000')
    type0001_dataset.to_excel(writer, sheet_name='type0001')
    type0002_dataset.to_excel(writer, sheet_name='type0002')
    type0003_dataset.to_excel(writer, sheet_name='type0003')
    type0004_dataset.to_excel(writer, sheet_name='type0004')
    type0005_dataset.to_excel(writer, sheet_name='type0005')
    type0006_dataset.to_excel(writer, sheet_name='type0006')
    type0007_dataset.to_excel(writer, sheet_name='type0007')
    type0008_dataset.to_excel(writer, sheet_name='type0008')
    type0009_dataset.to_excel(writer, sheet_name='type0009')
    type0010_dataset.to_excel(writer, sheet_name='type0010')
    type0011_dataset.to_excel(writer, sheet_name='type0011')

    workbook = writer.book
    cell_format = workbook.add_format({'text_wrap': True})

    for key, value in pickle_files.items():
        raw_dataset = pickle.load(open(value[0],"rb"))
        curr_type = value[0][4:12] 
        worksheet = writer.sheets[curr_type]
        worksheet.set_column('A:Z', cell_format=cell_format)